# ✈️ Enconectando  Postegree com Python usando sqlalchemy


Este notebook executa o processo completo de **ETL (Extract, Transform, Load)** dos dados públicos de ocorrências aeronáuticas da ANAC.

---


In [1]:
# ==============================================================================
# PASSO 1: LEITURA E EXTRAÇÃO DO ARQUIVO JSON
# ==============================================================================

# Importa a biblioteca 'pandas' e define o apelido padrão 'pd'
import pandas as pd

# Configura o pandas para exibir todas as colunas no resultado, sem ocultar com '...'
pd.set_option('display.max_columns', None)

# Armazena o caminho absoluto do arquivo JSON no seu computador
# O prefixo 'r' define uma raw string, evitando erros com caracteres de escape ou barras
caminho_do_arquivo = r"/Users/analeia/Desktop/Engenharia de Dados/V_OCORRENCIA_AMPLA.json"

# Carrega o arquivo JSON em memória como uma tabela chamada DataFrame (variável 'db')
# encoding='utf-8-sig' trata corretamente acentos e caracteres especiais da língua portuguesa
db = pd.read_json(caminho_do_arquivo, encoding='utf-8-sig')

# Exibe as 10 primeiras linhas da tabela para confirmar que os dados foram carregados
db.head(3)


,Numero_da_Ocorrencia,Numero_da_Ficha,Operador_Padronizado,Classificacao_da_Ocorrência,Data_da_Ocorrencia,Hora_da_Ocorrência,Municipio,UF,Regiao,Descricao_do_Tipo,ICAO,Latitude,Longitude,Tipo_de_Aerodromo,Historico,Matricula,Categoria_da_Aeronave,Operador,Tipo_de_Ocorrencia,Fase_da_Operacao,Operacao,Danos_a_Aeronave,Aerodromo_de_Destino,Aerodromo_de_Origem,Lesoes_Fatais_Tripulantes,Lesoes_Fatais_Passageiros,Lesoes_Fatais_Terceiros,Lesoes_Graves_Tripulantes,Lesoes_Graves_Passageiros,Lesoes_Graves_Terceiros,Lesoes_Leves_Tripulantes,Lesoes_Leves_Passageiros,Lesoes_Leves_Terceiros,Ilesos_Tripulantes,Ilesos_Passageiros,Lesoes_Desconhecidas_Tripulantes,Lesoes_Desconhecidas_Passageiros,Lesoes_Desconhecidas_Terceiros,Modelo,CLS,Tipo_ICAO,PMD,Numero_de_Assentos,Nome_do_Fabricante,PSSO
0,33947,2022722007,AZUL LINHAS AÉREAS BRASILEIRAS S.A.,Incidente,2022-08-11,05:31:00,PALMAS,TO,Norte,FALHA OU MAU FUNCIONAMENTO DE SISTEMA / COMPON...,SBPJ,NaN,NaN,Público,A aeronave decolou do aeródromo Marechal Cunha...,PRAYJ,TPR,AZUL LINHAS AÉREAS BRASILEIRAS S.A.,SCF-NP,Cruzeiro,Voo Regular,Nenhum,SBKP,SBSL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,ERJ 190-200 LR,L2J,E195,50790.0,125.0,EMBRAER,verdadeiro
1,29123,202140534,AZUL,Incidente Grave,2021-08-06,16:59,CAXIAS DO SUL,RS,Sul,CONTATO ANORMAL COM A PISTA,SBCX,"-29,1956","-51,1897",Público,A AERONAVE DECOLOU DO AERÓDROMO VIRACOPOS (SBK...,PRPJN,TPR,AZUL LINHAS AEREAS BRASILEIRAS S.A,ARC,Pouso,Voo Regular,Leve,SBCX,SBKP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,135.0,NaN,NaN,NaN,ERJ 190-400,L2J,E295,61500.0,143.0,EMBRAER,verdadeiro
2,32189,2022293670,FIGUEIREDO AVIAÇÃO AGRÍCOLA EIRELI,Incidente Grave,2022-05-21,12:25,DOURADOS,MS,Centro-Oeste,OPERAÇÃO A BAIXA ALTITUDE,Fora de Aeródromo,"-22,0411","-54,4103",-,A aeronave decolou do aeródromo Asas do MS (SS...,PTGGQ,S11,FIGUEIREDO AVIAÇÃO AGRÍCOLA EIRELI,LALT,Manobra,Operação Agrícola,Leve,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,EMB-201,L1P,IPAN,1800.0,1.0,NEIVA,verdadeiro


In [2]:
# ==============================================================================
# PASSO 2: INSPEÇÃO DAS COLUNAS ORIGINAIS
# ==============================================================================

# Lista os nomes de todas as colunas presentes no arquivo original da ANAC
# Esse passo é essencial para conferirmos a grafia exata de cada coluna antes de selecionar ou renomear
db.columns
db.head()

,Numero_da_Ocorrencia,Numero_da_Ficha,Operador_Padronizado,Classificacao_da_Ocorrência,Data_da_Ocorrencia,Hora_da_Ocorrência,Municipio,UF,Regiao,Descricao_do_Tipo,ICAO,Latitude,Longitude,Tipo_de_Aerodromo,Historico,Matricula,Categoria_da_Aeronave,Operador,Tipo_de_Ocorrencia,Fase_da_Operacao,Operacao,Danos_a_Aeronave,Aerodromo_de_Destino,Aerodromo_de_Origem,Lesoes_Fatais_Tripulantes,Lesoes_Fatais_Passageiros,Lesoes_Fatais_Terceiros,Lesoes_Graves_Tripulantes,Lesoes_Graves_Passageiros,Lesoes_Graves_Terceiros,Lesoes_Leves_Tripulantes,Lesoes_Leves_Passageiros,Lesoes_Leves_Terceiros,Ilesos_Tripulantes,Ilesos_Passageiros,Lesoes_Desconhecidas_Tripulantes,Lesoes_Desconhecidas_Passageiros,Lesoes_Desconhecidas_Terceiros,Modelo,CLS,Tipo_ICAO,PMD,Numero_de_Assentos,Nome_do_Fabricante,PSSO
0,33947,2022722007,AZUL LINHAS AÉREAS BRASILEIRAS S.A.,Incidente,2022-08-11,05:31:00,PALMAS,TO,Norte,FALHA OU MAU FUNCIONAMENTO DE SISTEMA / COMPON...,SBPJ,NaN,NaN,Público,A aeronave decolou do aeródromo Marechal Cunha...,PRAYJ,TPR,AZUL LINHAS AÉREAS BRASILEIRAS S.A.,SCF-NP,Cruzeiro,Voo Regular,Nenhum,SBKP,SBSL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,ERJ 190-200 LR,L2J,E195,50790.0,125.0,EMBRAER,verdadeiro
1,29123,202140534,AZUL,Incidente Grave,2021-08-06,16:59,CAXIAS DO SUL,RS,Sul,CONTATO ANORMAL COM A PISTA,SBCX,"-29,1956","-51,1897",Público,A AERONAVE DECOLOU DO AERÓDROMO VIRACOPOS (SBK...,PRPJN,TPR,AZUL LINHAS AEREAS BRASILEIRAS S.A,ARC,Pouso,Voo Regular,Leve,SBCX,SBKP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,135.0,NaN,NaN,NaN,ERJ 190-400,L2J,E295,61500.0,143.0,EMBRAER,verdadeiro
2,32189,2022293670,FIGUEIREDO AVIAÇÃO AGRÍCOLA EIRELI,Incidente Grave,2022-05-21,12:25,DOURADOS,MS,Centro-Oeste,OPERAÇÃO A BAIXA ALTITUDE,Fora de Aeródromo,"-22,0411","-54,4103",-,A aeronave decolou do aeródromo Asas do MS (SS...,PTGGQ,S11,FIGUEIREDO AVIAÇÃO AGRÍCOLA EIRELI,LALT,Manobra,Operação Agrícola,Leve,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,EMB-201,L1P,IPAN,1800.0,1.0,NEIVA,verdadeiro
3,32190,2022354545,FENIX PARTICIPACOES LTDA,Incidente,2022-02-12,15:30,CAMPINAS,SP,Sudeste,COLISÃO COM AVE,SDAM,"-22,8592","-47,1081",Público,A aeronave decolou do Aeródromo Estadual de Ca...,PRVZE,TPP,FENIX PARTICIPACOES LTDA,BIRD,Decolagem,Voo Privado,Leve,SBCY,SDAM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,NaN,NaN,NaN,PC-12/47E,L1T,PC12,4740.0,10.0,PILATUS,verdadeiro
4,32726,2022125796,ANAPE EMPREENDIMENTOS IMOBILIÁRIOS-EIRELI,Incidente,2022-06-26,13:45,CAMPINAS,SP,Sudeste,COLISÃO NO SOLO,SDAM,"-22,8592","-47,1081",Público,A aeronave decolou do heliponto Santuário Naci...,PRPIN,TPP,ANAPE EMPREENDIMENTOS IMOBILIÁRIOS-EIRELI,GCOL,Estacionamento,Voo Privado,Leve,SIYL,SIYL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3.0,NaN,NaN,NaN,A109E,H2T,A109,3000.0,8.0,AGUSTA,verdadeiro


In [3]:
db.dtypes

Numero_da_Ocorrencia                  int64
Numero_da_Ficha                         str
Operador_Padronizado                    str
Classificacao_da_Ocorrência             str
Data_da_Ocorrencia                      str
Hora_da_Ocorrência                      str
Municipio                               str
UF                                      str
Regiao                                  str
Descricao_do_Tipo                       str
ICAO                                    str
Latitude                                str
Longitude                               str
Tipo_de_Aerodromo                       str
Historico                               str
Matricula                               str
Categoria_da_Aeronave                   str
Operador                                str
Tipo_de_Ocorrencia                      str
Fase_da_Operacao                        str
Operacao                                str
Danos_a_Aeronave                        str
Aerodromo_de_Destino            

In [4]:
# ==============================================================================
# PASSO 3: SELEÇÃO DAS COLUNAS DE INTERESSE
# ==============================================================================

# Cria uma lista contendo exatamente as 7 colunas que queremos levar para o nosso banco de dados
colunas = [
    "Numero_da_Ocorrencia",         # Identificador único da ocorrência aeronáutica
    "Classificacao_da_Ocorrência",  # Categoria da gravidade (ainda com acento na origem)
    "Data_da_Ocorrencia",           # Data em que o fato aconteceu
    "Municipio",                    # Município/cidade da ocorrência
    "UF",                           # Estado / Unidade da Federação
    "Regiao",                       # Região do Brasil (ex.: Sudeste, Sul, Norte...)
    "Nome_do_Fabricante"            # Fabricante da aeronave (ex.: Embraer, Boeing...)
]

# Proteção contra reexecução: caso você execute esta célula novamente após já ter rodado
# a Célula 4, a coluna com acento não existirá mais. Esta checagem evita o erro 'KeyError':
if 'Classificacao_da_Ocorrência' not in db.columns and 'Classificacao_da_Ocorrencia' in db.columns:
    colunas[1] = 'Classificacao_da_Ocorrencia'

# Filtra o DataFrame 'db' mantendo apenas as 7 colunas da lista acima e descartando o resto
db = db[colunas]


In [5]:
# ==============================================================================
# PASSO 4: TRATAMENTO E PADRONIZAÇÃO DE NOMES (REMOVER ACENTOS)
# ==============================================================================

# Boas práticas em Engenharia de Dados: nunca use acentos ou espaços em nomes de colunas no banco!
# Aqui renomeamos 'Classificacao_da_Ocorrência' (com circunflexo) para 'Classificacao_da_Ocorrencia' (sem acento)
if 'Classificacao_da_Ocorrência' in db.columns:
    db = db.rename(columns={'Classificacao_da_Ocorrência': 'Classificacao_da_Ocorrencia'})

# ==============================================================================
# PASSO 5: VALIDAÇÃO VISUAL DOS DADOS TRATADOS
# ==============================================================================

# Exibe as 3 primeiras linhas do DataFrame já filtrado e com as colunas padronizadas
# Observe que agora a coluna se chama 'Classificacao_da_Ocorrencia' (sem acento)
db.head(3)


,Numero_da_Ocorrencia,Classificacao_da_Ocorrencia,Data_da_Ocorrencia,Municipio,UF,Regiao,Nome_do_Fabricante
0,33947,Incidente,2022-08-11,PALMAS,TO,Norte,EMBRAER
1,29123,Incidente Grave,2021-08-06,CAXIAS DO SUL,RS,Sul,EMBRAER
2,32189,Incidente Grave,2022-05-21,DOURADOS,MS,Centro-Oeste,NEIVA


In [6]:
db.dtypes

Numero_da_Ocorrencia           int64
Classificacao_da_Ocorrencia      str
Data_da_Ocorrencia               str
Municipio                        str
UF                               str
Regiao                           str
Nome_do_Fabricante               str
dtype: object

In [7]:
import psycopg2
from sqlalchemy import create_engine

# Parâmetros de conexão
dbname = 'python'
user = 'analeia'
password = '12345'
host = 'localhost'
port = '5432'

# Adicionamos +psycopg2 para avisar qual biblioteca usar:
conn = f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}'

# Cria o motor de conexão do SQLAlchemy
engine = create_engine(conn)

# Envia os dados para a tabela no PostgreSQL!
nome_tabela = 'teste'
db.to_sql(nome_tabela, con=engine, if_exists='replace', index=False)

print(f"✅ Tabela '{nome_tabela}' criada e carregada com sucesso!")

✅ Tabela 'teste' criada e carregada com sucesso!


In [8]:
# ==============================================================================
# PASSO 6: CARGA COMPLETA NO BANCO DE DADOS POSTGRESQL (COM LIMPEZA PRÉVIA)
# ==============================================================================

# Importa a biblioteca 'psycopg2' (driver conector entre Python e PostgreSQL)
import psycopg2

# ------------------------------------------------------------------------------
# 6.1 PARÂMETROS DE CONEXÃO COM O BANCO DE DADOS
# ------------------------------------------------------------------------------
dbname = 'python'      # Nome do banco de dados no PostgreSQL
user = 'analeia'       # Usuário cadastrado no PostgreSQL
password = '12345'     # Senha do usuário
host = 'localhost'     # Endereço do servidor ('localhost' = máquina local)
port = '5432'          # Porta padrão do serviço PostgreSQL

# ------------------------------------------------------------------------------
# 6.2 ABERTURA DA CONEXÃO E CRIAÇÃO DO CURSOR
# ------------------------------------------------------------------------------
# Estabelece o canal de comunicação física com o banco de dados PostgreSQL
conn = psycopg2.connect(
    dbname=dbname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Cria o cursor: objeto responsável por enviar comandos SQL e buscar respostas no banco
cur = conn.cursor()

# ------------------------------------------------------------------------------
# 6.3 GARANTIR QUE A TABELA EXISTE ANTES DA CARGA
# ------------------------------------------------------------------------------
# Cria a tabela 'Anac' se ela ainda não existir, definindo os tipos de dados de cada coluna
cur.execute("""
CREATE TABLE IF NOT EXISTS Anac (
    Numero_da_Ocorrencia INT,
    Classificacao_da_Ocorrencia VARCHAR(50),
    Data_da_Ocorrencia DATE,
    Municipio VARCHAR(50),
    UF VARCHAR(30),
    Regiao VARCHAR(30),
    Nome_do_Fabricante VARCHAR(100)
);
""")

# ------------------------------------------------------------------------------
# 6.4 DELETE / TRUNCATE DA BASE ANTES DA CARGA (LIMPEZA PRÉVIA)
# ------------------------------------------------------------------------------
# Esse comando limpa toda a tabela antes de iniciar a nova inserção.
# Isso impede que os dados fiquem se acumulando/duplicando a cada execução do script.
# Usamos TRUNCATE TABLE porque é instantâneo e muito mais rápido que DELETE FROM em lote.
cur.execute("TRUNCATE TABLE Anac;")

# ------------------------------------------------------------------------------
# 6.5 CARGA DOS DADOS: INSERINDO TODAS AS LINHAS DO DATAFRAME
# ------------------------------------------------------------------------------
# db.iterrows() percorre cada uma das 12.221 linhas da tabela:
# 'indice' representa o índice da linha (0, 1, 2...)
# 'coluna' representa os dados daquela linha acessíveis pelo nome da coluna
for indice, coluna in db.iterrows():
    cur.execute("""
    INSERT INTO Anac (
        Numero_da_Ocorrencia,
        Classificacao_da_Ocorrencia,
        Data_da_Ocorrencia,
        Municipio,
        UF,
        Regiao,
        Nome_do_Fabricante
    ) VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (
        coluna['Numero_da_Ocorrencia'],        # %s 1: Inteiro
        coluna['Classificacao_da_Ocorrencia'],  # %s 2: Texto
        coluna['Data_da_Ocorrencia'],           # %s 3: Data (AAAA-MM-DD)
        coluna['Municipio'],                    # %s 4: Texto
        coluna['UF'],                           # %s 5: Texto (2 letras)
        coluna['Regiao'],                       # %s 6: Texto
        coluna['Nome_do_Fabricante']            # %s 7: Texto
    ))

# ------------------------------------------------------------------------------
# 6.6 EFETIVAÇÃO DAS ALTERAÇÕES NO BANCO (COMMIT)
# ------------------------------------------------------------------------------
# Grava permanentemente todas as 12.221 inserções no PostgreSQL
conn.commit()

# ------------------------------------------------------------------------------
# 6.7 VALIDAÇÃO AUTOMÁTICA DA QUANTIDADE DE LINHAS
# ------------------------------------------------------------------------------
# Consulta o banco de dados para checar quantas linhas estão salvas na tabela Anac
cur.execute("SELECT COUNT(*) FROM Anac;")
total_banco = cur.fetchone()[0]


# ------------------------------------------------------------------------------
# 6.8 FECHAMENTO SEGURO DOS RECURSOS
# ------------------------------------------------------------------------------
# Fecha o cursor para liberar os recursos de memória do servidor
cur.close()

# Fecha a conexão com o banco de dados PostgreSQL
conn.close()
